In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🏢 Procurement Approval Copilot

## What We're Building

A procurement review workflow:
1. **Extract vendor details** — parse vendor proposal
2. **Risk flags** — identify red flags and concerns
3. **Approval summary** — create executive summary
4. **Final recommendation** — approve / conditional / reject

## Architecture

```
Vendor Proposal
      │
      ▼
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────────┐
│ Extract  │──▶│ Risk     │──▶│ Approval │──▶│ Recommend    │
│ Details  │   │ Flags    │   │ Summary  │   │ (conditional)│
└──────────┘   └──────────┘   └──────────┘   └──────────────┘
```

## Key LangGraph Concept: Conditional End States

The recommendation node routes to different final actions based on
the risk assessment — auto-approve low-risk, escalate high-risk.

## Stack
- **LangGraph** — pipeline with conditional final step
- **Ollama** — local LLM

## Step 1 — Imports & Setup

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)
print("All imports successful!")

## Step 2 — State & Sample Vendor Proposals

In [ ]:
class ProcurementState(TypedDict):
    proposal_text: str
    vendor_details: str        # Extracted structured data
    risk_assessment: str       # Identified risks
    risk_level: str            # low / medium / high
    approval_summary: str      # Executive summary
    recommendation: str        # Final recommendation
    action: str                # approved / conditional / escalated


proposals = {
    "low_risk": """
VENDOR PROPOSAL: Office Supplies Contract

Vendor: OfficeMax Solutions (est. 1995)
Contract Value: $24,000/year ($2,000/month)
Duration: 12 months with 30-day cancellation
Payment: Net-30, standard invoicing
References: 3 provided (Fortune 500 clients)
Insurance: $2M general liability, current certificate attached
Compliance: SOC 2 Type II certified, US-based operations
Delivery: Free shipping on orders >$200, 2-day delivery guaranteed
Discount: 15% below list price on all items
""",

    "high_risk": """
VENDOR PROPOSAL: Cloud Infrastructure Migration

Vendor: CloudFirst Technologies (est. 2022)
Contract Value: $2,400,000 over 36 months
Duration: 36 months, minimum commitment, early termination fee = 100% remaining
Payment: 50% upfront ($1.2M), remainder monthly
References: 2 provided (both startups, one went bankrupt)
Insurance: $500K general liability only
Compliance: No SOC 2 certification, "in progress"
Data handling: Client data stored in EU and Singapore
IP: All migration scripts and tooling remain CloudFirst property
SLA: 99.5% uptime (industry standard is 99.99%)
Team: 3 engineers assigned (for a 36-month enterprise migration)
Subcontracting: CloudFirst may subcontract up to 80% of work
"""
}

print(f"Loaded {len(proposals)} sample proposals")

## Step 3 — Define Nodes

In [ ]:
# --- Node 1: Extract vendor details ---
extract_prompt = ChatPromptTemplate.from_template(
    """Extract structured details from this vendor proposal.

Proposal:
{proposal}

Extract these fields:
- Vendor Name & Year Established
- Contract Value (total and monthly)
- Contract Duration
- Payment Terms
- References Summary
- Insurance Coverage
- Compliance/Certifications
- Data Handling / Privacy
- IP Terms
- SLA / Service Guarantees
- Cancellation Terms

Structured details:"""
)


def extract_details(state: ProcurementState) -> dict:
    print("📋 Node: extract_details")
    chain = extract_prompt | llm | StrOutputParser()
    details = chain.invoke({"proposal": state["proposal_text"]})
    return {"vendor_details": details}


# --- Node 2: Risk assessment ---
risk_prompt = ChatPromptTemplate.from_template(
    """You are a procurement risk analyst. Assess this vendor proposal for risks.

Vendor Details:
{details}

Original Proposal:
{proposal}

For each risk found, provide:
- RISK: Description
- SEVERITY: low / medium / high / critical
- MITIGATION: How to address it

Common risk areas to check:
- Financial (upfront payments, termination fees, vendor stability)
- Operational (SLA inadequacy, understaffing, subcontracting)
- Legal (IP ownership, data sovereignty, compliance gaps)
- Reputational (reference quality, vendor age/track record)

End with: OVERALL RISK LEVEL: low / medium / high

Risk assessment:"""
)


def assess_risk(state: ProcurementState) -> dict:
    print("⚠️ Node: assess_risk")
    chain = risk_prompt | llm | StrOutputParser()
    assessment = chain.invoke({"details": state["vendor_details"], "proposal": state["proposal_text"]})

    risk_level = "medium"  # default
    for level in ["high", "medium", "low"]:
        if f"OVERALL RISK LEVEL: {level}" in assessment.lower() or f"overall risk level: {level}" in assessment.lower():
            risk_level = level
            break
    # Also check for high-risk keywords
    assessment_lower = assessment.lower()
    if "critical" in assessment_lower and assessment_lower.count("critical") >= 2:
        risk_level = "high"

    print(f"     Risk level: {risk_level}")
    return {"risk_assessment": assessment, "risk_level": risk_level}


# --- Node 3: Approval summary ---
summary_prompt = ChatPromptTemplate.from_template(
    """Create an executive approval summary for this procurement request.

Vendor Details:
{details}

Risk Assessment:
{risks}

Write a concise summary (5-8 sentences) suitable for a VP/CFO to read.
Include: what we're buying, from whom, key terms, key risks, and a recommendation.

Executive summary:"""
)


def create_summary(state: ProcurementState) -> dict:
    print("📊 Node: create_summary")
    chain = summary_prompt | llm | StrOutputParser()
    summary = chain.invoke({"details": state["vendor_details"], "risks": state["risk_assessment"]})
    return {"approval_summary": summary}


# --- Node 4a: Auto-approve (low risk) ---
def auto_approve(state: ProcurementState) -> dict:
    print("  ✅ Auto-approved (low risk)")
    return {
        "recommendation": "APPROVED — Low-risk vendor with standard terms. Proceed with contract signing.",
        "action": "approved"
    }


# --- Node 4b: Conditional approval (medium risk) ---
conditional_prompt = ChatPromptTemplate.from_template(
    """This procurement has medium risk. Write conditions that must be met
before approval can be granted.

Risk Assessment:
{risks}

List specific conditions (negotiate X, require Y, add clause Z):

Conditions for approval:"""
)


def conditional_approve(state: ProcurementState) -> dict:
    print("  ⚡ Conditional approval (medium risk)")
    chain = conditional_prompt | llm | StrOutputParser()
    conditions = chain.invoke({"risks": state["risk_assessment"]})
    return {
        "recommendation": f"CONDITIONAL APPROVAL — The following conditions must be met:\n{conditions}",
        "action": "conditional"
    }


# --- Node 4c: Escalate (high risk) ---
def escalate_procurement(state: ProcurementState) -> dict:
    print("  🚨 Escalated to leadership (high risk)")
    return {
        "recommendation": (
            f"ESCALATED — High-risk proposal requires VP/CFO review.\n"
            f"Key concerns:\n{state['risk_assessment'][:300]}..."
        ),
        "action": "escalated"
    }


print("All nodes defined")

## Step 4 — Build Graph with Risk-Based Routing

In [ ]:
def route_by_risk(state: ProcurementState) -> str:
    risk = state.get("risk_level", "medium").lower()
    if "low" in risk:
        return "auto_approve"
    elif "high" in risk:
        return "escalate"
    return "conditional"


workflow = StateGraph(ProcurementState)
workflow.add_node("extract_details", extract_details)
workflow.add_node("assess_risk", assess_risk)
workflow.add_node("create_summary", create_summary)
workflow.add_node("auto_approve", auto_approve)
workflow.add_node("conditional", conditional_approve)
workflow.add_node("escalate", escalate_procurement)

workflow.set_entry_point("extract_details")
workflow.add_edge("extract_details", "assess_risk")
workflow.add_edge("assess_risk", "create_summary")

workflow.add_conditional_edges("create_summary", route_by_risk, {
    "auto_approve": "auto_approve",
    "conditional": "conditional",
    "escalate": "escalate",
})

workflow.add_edge("auto_approve", END)
workflow.add_edge("conditional", END)
workflow.add_edge("escalate", END)

app = workflow.compile()
print("Procurement copilot compiled with risk-based routing")

## Step 5 — Process Proposals

In [ ]:
def process_proposal(name: str, proposal_text: str):
    print(f"\n{'='*60}")
    print(f"🏢 Processing: {name}")
    print("="*60)

    result = app.invoke({
        "proposal_text": proposal_text,
        "vendor_details": "", "risk_assessment": "", "risk_level": "",
        "approval_summary": "", "recommendation": "", "action": "",
    })

    print(f"\n📊 Executive Summary:")
    print(result["approval_summary"])
    print(f"\n📌 Recommendation ({result['action'].upper()}):")
    print(result["recommendation"])
    return result


# Low-risk proposal
r1 = process_proposal("Office Supplies (low risk)", proposals["low_risk"])

In [ ]:
# High-risk proposal
r2 = process_proposal("Cloud Migration (high risk)", proposals["high_risk"])

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Detail extraction** | LLM parses unstructured proposal into structured fields |
| **Risk scoring** | LLM evaluates financial, operational, legal risks |
| **Conditional routing** | Route to approve/conditional/escalate based on risk |
| **Executive summary** | Concise briefing for decision-makers |

## 🔧 Extensions

- **Vendor database**: Check vendor against known vendors, past performance
- **Budget check**: Compare proposal cost against department budget
- **Multi-approver**: Route to different approvers based on dollar amount
- **Contract redlining**: Auto-generate contract markup with risk mitigations